# Writing Manning's N Values to Land Cover Sidecar HDF

Modify Manning's N values in HEC-RAS land cover sidecar HDF files with
automatic version detection, backup safety, and round-trip verification.
Supports HEC-RAS v5.0.7 through v6.6+.


In [ ]:
from pathlib import Path
import logging
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from ras_commander import *
from ras_commander.hdf import HdfLandCover

for logger_name in [
    'ras_commander.RasExamples',
    'ras_commander.RasMap',
    'ras_commander.RasPrj',
    'ras_commander.RasUtils',
    'ras_commander.hdf.HdfBase',
    'ras_commander.hdf.HdfLandCover',
]:
    logging.getLogger(logger_name).setLevel(logging.WARNING)


## Setup & Project Extraction

Use the reproducible `BaldEagleCrkMulti2D` example project, initialize the
project metadata, and locate the land cover sidecar HDF that stores the base
Manning's N lookup table.


In [ ]:
RAS_VERSION = '7.0'
TARGET_UPDATES = {
    'Deciduous Forest': 0.130,
    'Developed, High Intensity': 0.180,
}

project_path = RasExamples.extract_project('BaldEagleCrkMulti2D')
init_ras_project(project_path, RAS_VERSION)

geometry_hdf_path = Path(
    ras.geom_df[ras.geom_df['has_2d_mesh']].iloc[0]['hdf_path']
)
landcover_hdf_path = Path(ras.rasmap_df.loc[0, 'landcover_hdf_path'][0])
backup_path = Path(str(landcover_hdf_path) + '.bak')

assert landcover_hdf_path.exists(), 'Expected land cover sidecar HDF.'

print(f"Project: {project_path.name}")
print(f"Geometry HDF: {geometry_hdf_path.name}")
print(f"Land cover sidecar: {landcover_hdf_path.name}")
print(f"Backup file: {backup_path.name}")


## Read Current Values

`get_landcover_raster_map()` reads the sidecar lookup table as a DataFrame,
including the raster pixel ID, class name, and current Manning's N value.


In [ ]:
raster_map_before = HdfLandCover.get_landcover_raster_map(landcover_hdf_path)
assert raster_map_before is not None and len(raster_map_before) > 0

target_before = (
    raster_map_before[raster_map_before['class_name'].isin(TARGET_UPDATES.keys())]
    .copy()
    .sort_values('class_name')
    .reset_index(drop=True)
)

expected_baseline = pd.Series(
    {
        'Deciduous Forest': 0.100,
        'Developed, High Intensity': 0.150,
    }
).sort_index()

baseline_check = target_before.set_index('class_name')['mannings_n'].sort_index()
np.testing.assert_allclose(baseline_check.values, expected_baseline.values)

print(f"Raster classes returned: {len(raster_map_before)}")
display(
    raster_map_before[['pixel_value', 'class_name', 'mannings_n']]
    .sort_values('pixel_value')
    .reset_index(drop=True)
)
print('Target classes used in this notebook:')
display(target_before)


## Detect Sidecar Format

`_detect_sidecar_format()` identifies which internal layout the sidecar uses.
That lets `set_landcover_raster_map()` update the correct datasets for older
v5 files, HEC-RAS 6.0 files, and modern 6.x files.


In [ ]:
sidecar_format = HdfLandCover._detect_sidecar_format(landcover_hdf_path)
known_formats = {'v5', 'v6_0', 'v6_modern'}

print(f"Detected sidecar format: {sidecar_format}")
assert sidecar_format in known_formats


## Modify Manning's N

Update two classes in the sidecar HDF to represent a rougher floodplain:
`Deciduous Forest` from 0.100 to 0.130 and `Developed, High Intensity`
from 0.150 to 0.180.


In [ ]:
write_result = HdfLandCover.set_landcover_raster_map(
    landcover_hdf_path,
    TARGET_UPDATES,
)
assert write_result['format'] == sidecar_format
assert write_result['changed'] == len(TARGET_UPDATES)

class_details_df = (
    pd.DataFrame(write_result['class_details'])
    .query('changed')
    .sort_values('class_name')
    .reset_index(drop=True)
)

raster_map_after = HdfLandCover.get_landcover_raster_map(landcover_hdf_path)
target_after = (
    raster_map_after[raster_map_after['class_name'].isin(TARGET_UPDATES.keys())]
    .copy()
    .sort_values('class_name')
    .reset_index(drop=True)
)

compare_df = (
    target_before[['class_name', 'mannings_n']]
    .rename(columns={'mannings_n': 'before_n'})
    .merge(
        target_after[['class_name', 'mannings_n']].rename(
            columns={'mannings_n': 'after_n'}
        ),
        on='class_name',
        how='inner',
    )
)
compare_df['delta_n'] = compare_df['after_n'] - compare_df['before_n']

print(f"Classes updated: {write_result['changed']}")
print(f"Backup created: {backup_path.name}")
display(
    class_details_df[
        ['class_name', 'old_mannings_n', 'new_mannings_n', 'value_changed']
    ]
)

fig, ax = plt.subplots(figsize=(8, 4))
compare_df.set_index('class_name')[['before_n', 'after_n']].plot(
    kind='bar',
    ax=ax,
    color=['#7aa6c2', '#d97b66'],
)
ax.set_ylabel("Manning's n")
ax.set_title("Before vs after sidecar updates")
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

display(compare_df)


## Round-Trip Verification

Read the sidecar back after writing and confirm the stored values match the
requested updates exactly.


In [ ]:
round_trip_map = HdfLandCover.get_landcover_raster_map(landcover_hdf_path)
round_trip_df = (
    round_trip_map[
        round_trip_map['class_name'].isin(TARGET_UPDATES.keys())
    ]
    .copy()
    .sort_values('class_name')
    .reset_index(drop=True)
)
round_trip_df = round_trip_df.rename(columns={'mannings_n': 'written_value'})
round_trip_df['expected_value'] = round_trip_df['class_name'].map(TARGET_UPDATES)
round_trip_df['matches_written_value'] = np.isclose(
    round_trip_df['written_value'],
    round_trip_df['expected_value'],
)

assert round_trip_df['matches_written_value'].all()
display(round_trip_df)


## Backup Verification

`set_landcover_raster_map()` creates a `.bak` copy before modifying the file.
Compare the backup against the modified sidecar to confirm the original values
are preserved.


In [ ]:
assert backup_path.exists(), 'Expected .bak file after write operation.'

backup_map = HdfLandCover.get_landcover_raster_map(backup_path)
backup_targets = (
    backup_map[backup_map['class_name'].isin(TARGET_UPDATES.keys())]
    .copy()
    .sort_values('class_name')
    .reset_index(drop=True)
)

backup_compare_df = (
    backup_targets[['class_name', 'mannings_n']]
    .rename(columns={'mannings_n': 'backup_n'})
    .merge(
        target_after[['class_name', 'mannings_n']].rename(
            columns={'mannings_n': 'modified_n'}
        ),
        on='class_name',
        how='inner',
    )
)
backup_compare_df['delta_n'] = (
    backup_compare_df['modified_n'] - backup_compare_df['backup_n']
)

display(backup_compare_df)


## Restore from Backup

Restoring is a normal file copy operation: copy the `.bak` file back over the
sidecar HDF, then re-read the values to confirm the original Manning's N table
is back in place.


In [ ]:
shutil.copy2(backup_path, landcover_hdf_path)

restored_map = HdfLandCover.get_landcover_raster_map(landcover_hdf_path)
restored_targets = (
    restored_map[restored_map['class_name'].isin(TARGET_UPDATES.keys())]
    .copy()
    .sort_values('class_name')
    .reset_index(drop=True)
)

restore_check_df = (
    target_before[['class_name', 'mannings_n']]
    .rename(columns={'mannings_n': 'original_n'})
    .merge(
        restored_targets[['class_name', 'mannings_n']].rename(
            columns={'mannings_n': 'restored_n'}
        ),
        on='class_name',
        how='inner',
    )
)
restore_check_df['restored_matches_original'] = np.isclose(
    restore_check_df['original_n'],
    restore_check_df['restored_n'],
)

assert restore_check_df['restored_matches_original'].all()
display(restore_check_df)


## Calibration Workflow Example

A typical calibration loop adjusts a few roughness classes, runs the plan,
reviews results, then restores the backup before trying the next candidate.
This table shows three simple trial sets you could evaluate in sequence.


In [ ]:
calibration_trials = pd.DataFrame(
    [
        {
            'trial_name': 'Baseline',
            'Deciduous Forest': float(
                target_before.set_index('class_name').loc[
                    'Deciduous Forest', 'mannings_n'
                ]
            ),
            'Developed, High Intensity': float(
                target_before.set_index('class_name').loc[
                    'Developed, High Intensity', 'mannings_n'
                ]
            ),
        },
        {
            'trial_name': 'Intermediate check',
            'Deciduous Forest': 0.115,
            'Developed, High Intensity': 0.165,
        },
        {
            'trial_name': 'Rougher floodplain',
            'Deciduous Forest': 0.130,
            'Developed, High Intensity': 0.180,
        },
    ]
).set_index('trial_name')

display(calibration_trials)
print(
    'Typical loop: restore from .bak -> apply one trial with '
    'set_landcover_raster_map() -> compute plan -> compare results -> '
    'keep the best candidate.'
)


## Key Takeaways

- `get_landcover_raster_map()` gives a clean class-to-Manning's-N table for inspection.
- `_detect_sidecar_format()` lets the write path adapt to older and newer HEC-RAS sidecar layouts.
- `set_landcover_raster_map()` creates a `.bak` file before writing, so changes are reversible.
- A simple read-back check confirms the modified values were stored correctly.
- This pattern fits iterative calibration workflows where you adjust roughness, recompute, and restore as needed.
